# Task 9: Validation of SSA and ODE Implementations

Three validation tests:

1. **Convergence** — SSA moments → ODE/analytical values as $N_{\text{rep}} \to \infty$
2. **Steady-state match** — SSA vs exact analytical formulas across parameter regimes
3. **Edge cases** — degenerate parameters ($k_{\text{off}} = 0$, $k_{\text{deg}} = 0$)

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import plotly.graph_objects as go

from src.ssa_simulation import simulate_telegraph, compute_sample_moments
from src.ode_moments import solve_ode_moments
from src.ssa_visualization import (
    analytical_steady_state, show_sample_moments,
    _base_layout, _BLUE, _RED, _PURPLE, _FONT,
)

### Why step-indexed moments are biased

The function `compute_sample_moments` computes statistics at each
**step index** (reaction event number), not at each **time point**.
In the Gillespie SSA, events occur at a rate equal to the total
propensity $a_0$. When the gene is ON, $a_0$ is much higher
(due to the $k_{\text{syn}}$ contribution), so **more events happen
per unit time** when $G = 1$.

This means step-indexed averages sample the ON state more frequently
than the OFF state, creating a **systematic bias** that does not
decrease with $N_{\text{rep}}$.

For validation, we use **time-grid moments**: interpolating each
trajectory to a common time grid before averaging, which gives
unbiased estimates.

In [ ]:
def compute_time_grid_moments(data, n_points=500):
    """Compute moments on a fixed time grid (unbiased estimator).

    Instead of averaging at step indices, this interpolates each
    trajectory to a common set of time points using piece-wise
    constant interpolation (appropriate for CTMC trajectories),
    then averages across trajectories.
    """
    n_rep = data.shape[1]

    # Common time range: from latest start to earliest end
    t_max = data[-1, :, 0].min()  # earliest final time
    t_grid = np.linspace(0, t_max * 0.98, n_points)  # 2% margin

    G_grid = np.zeros((n_points, n_rep))
    R_grid = np.zeros((n_points, n_rep))

    for j in range(n_rep):
        t_j = data[:, j, 0]
        G_j = data[:, j, 1]
        R_j = data[:, j, 2]
        # Piece-wise constant: state just before each grid point
        idx = np.searchsorted(t_j, t_grid, side='right') - 1
        idx = np.clip(idx, 0, len(t_j) - 1)
        G_grid[:, j] = G_j[idx]
        R_grid[:, j] = R_j[idx]

    mu_G = G_grid.mean(axis=1)
    mu_R = R_grid.mean(axis=1)

    return {
        "time": t_grid,
        "mu_G": mu_G,
        "mu_R": mu_R,
        "sigma_G": G_grid.std(axis=1, ddof=1),
        "sigma_R": R_grid.std(axis=1, ddof=1),
        "cov_RG": np.array([
            np.cov(G_grid[i], R_grid[i])[0, 1]
            for i in range(n_points)
        ]),
    }


def tail_average(moments, tail_frac=0.1):
    """Steady-state estimate from the last tail_frac of a time-series."""
    n = len(moments['time'])
    tail = max(1, int(n * tail_frac))
    return {
        k: float(moments[k][-tail:].mean())
        for k in ["mu_G", "mu_R", "sigma_G", "sigma_R", "cov_RG"]
    }

---
## 1. Convergence: SSA → Analytical as $N_{\text{rep}} \to \infty$

Using time-grid moments, the steady-state error of the sample mean
should decrease as $\sigma / \sqrt{N_{\text{rep}}}$.

In [ ]:
kin = dict(k_on=0.1, k_off=0.1, k_syn=10.0, k_deg=0.2)
ss_exact = analytical_steady_state(**kin)

print(f'Analytical: μ_G = {ss_exact["mu_G"]:.3f}, '
      f'μ_R = {ss_exact["mu_R"]:.1f}, '
      f'Cov = {ss_exact["cov_RG"]:.4f}')

n_sim = 5000
n_rep_values = [50, 100, 200, 500, 1000]

errors_mu_G = []
errors_mu_R = []
errors_cov  = []

for n_rep in n_rep_values:
    data = simulate_telegraph(
        **kin, t0=0, g0=0, r0=0, n_sim=n_sim, n_rep=n_rep
    )
    m = compute_time_grid_moments(data, n_points=500)
    ss = tail_average(m)

    errors_mu_G.append(abs(ss['mu_G'] - ss_exact['mu_G']))
    errors_mu_R.append(abs(ss['mu_R'] - ss_exact['mu_R']))
    errors_cov.append(abs(ss['cov_RG'] - ss_exact['cov_RG']))

    print(
        f'  n_rep={n_rep:5d}  '
        f'|Δμ_G|={errors_mu_G[-1]:.4f}  '
        f'|Δμ_R|={errors_mu_R[-1]:.3f}  '
        f'|ΔCov|={errors_cov[-1]:.4f}'
    )

In [ ]:
# ── Convergence plot ──
fig = go.Figure(layout=_base_layout(
    title='<b>Convergence:</b> |SSA − Analytical| vs N<sub>rep</sub>',
    ylabel='Absolute error',
    figsize=(5.5, 3.2),
))
fig.update_xaxes(title_text='N<sub>rep</sub>', type='log')
fig.update_yaxes(title_text='Absolute error', type='log')

for err, name, col, sym in [
    (errors_mu_R, '|Δμ<sub>R</sub>|', _RED, 'circle'),
    (errors_mu_G, '|Δμ<sub>G</sub>|', _BLUE, 'square'),
    (errors_cov,  '|ΔCov|',           _PURPLE, 'diamond'),
]:
    fig.add_trace(go.Scatter(
        x=n_rep_values, y=err, mode='lines+markers',
        marker=dict(size=9, symbol=sym),
        line=dict(color=col, width=2),
        name=name,
    ))

# 1/√N reference
n_arr = np.array(n_rep_values, dtype=float)
ref = errors_mu_R[0] * np.sqrt(n_rep_values[0]) / np.sqrt(n_arr)
fig.add_trace(go.Scatter(
    x=n_rep_values, y=ref, mode='lines',
    line=dict(color='#94a3b8', width=1.5, dash='dot'),
    name='∝ 1/√N',
))

fig.show()

> **Pass criterion:** Error curves should roughly follow the grey
> $1/\sqrt{N}$ reference slope.

---
## 2. Steady-State Moments vs Analytical Formulas

| Moment | Formula |
|---|---|
| $\mu_G$ | $k_{\text{on}} / (k_{\text{on}} + k_{\text{off}})$ |
| $\mu_R$ | $k_{\text{syn}} \mu_G / k_{\text{deg}}$ |
| $\sigma_G^2$ | $\mu_G (1 - \mu_G)$ |
| $C_{RG}$ | $k_{\text{syn}} \sigma_G^2 / (k_{\text{on}} + k_{\text{off}} + k_{\text{deg}})$ |
| Fano | $1 + k_{\text{syn}} (1 - \mu_G) / (k_{\text{on}} + k_{\text{off}} + k_{\text{deg}})$ |

In [ ]:
test_cases = {
    "Slow switching": dict(k_on=0.01, k_off=0.02, k_syn=10.0, k_deg=0.2),
    "Fast switching": dict(k_on=2.0,  k_off=4.0,  k_syn=10.0, k_deg=0.2),
    "High expression": dict(k_on=0.1, k_off=0.1, k_syn=40.0, k_deg=0.5),
    "Low expression":  dict(k_on=0.1, k_off=0.1, k_syn=2.0,  k_deg=1.0),
}

all_results = []

for name, kin in test_cases.items():
    data = simulate_telegraph(
        **kin, t0=0, g0=0, r0=0, n_sim=6000, n_rep=500
    )
    m = compute_time_grid_moments(data, n_points=500)
    ss = tail_average(m)
    ana = analytical_steady_state(**kin)
    all_results.append((name, ss, ana))

    print(f"\n── {name} ──")
    print(f"  {"Moment":<10s}  {"SSA":>10s}  {"Analytical":>10s}  {"Rel.Err":>8s}")
    print(f"  {"-"*10}  {"-"*10}  {"-"*10}  {"-"*8}")
    for key in ["mu_G", "mu_R", "sigma_G", "sigma_R", "cov_RG"]:
        s_val = ss[key]
        a_val = ana[key]
        rel = abs(s_val - a_val) / max(abs(a_val), 1e-10)
        status = "✅" if rel < 0.15 else "⚠️"
        print(f"  {key:<10s}  {s_val:10.4f}  {a_val:10.4f}  {rel:7.1%} {status}")

In [ ]:
# Visual: grouped bar chart
fig = go.Figure(layout=_base_layout(
    title='<b>Steady-state:</b> SSA vs Analytical',
    ylabel='Value',
    figsize=(6.0, 3.5),
))

regimes = [r[0] for r in all_results]

for moment, color in [
    ('mu_G', _BLUE), ('mu_R', _RED), ('cov_RG', _PURPLE)
]:
    ssa_vals = [r[1][moment] for r in all_results]
    ana_vals = [r[2][moment] for r in all_results]

    fig.add_trace(go.Bar(
        x=regimes, y=ssa_vals,
        name=f'{moment} (SSA)',
        marker_color=color, opacity=0.7,
    ))
    fig.add_trace(go.Bar(
        x=regimes, y=ana_vals,
        name=f'{moment} (analytical)',
        marker_color=color,
        marker_pattern_shape='/',
    ))

fig.update_layout(barmode='group')
fig.update_xaxes(title_text='')
fig.show()

---
## 3. Edge Cases

### 3a. Constitutive expression ($k_{\text{off}} = 0$, $G_0 = 1$)

Gene never turns off → pure birth–death.
Expected: $\langle G \rangle = 1$, $\langle R \rangle = 50$, Fano = 1, $\text{Cov} = 0$.

In [ ]:
data_const = simulate_telegraph(
    k_on=0.1, k_off=0.0, k_syn=10.0, k_deg=0.2,
    t0=0.0, g0=1, r0=0, n_sim=4000, n_rep=500,
)
m_const = compute_time_grid_moments(data_const)
ss_c = tail_average(m_const)

print('── Constitutive (k_off = 0) ──')
print(f'  ⟨G⟩  = {ss_c["mu_G"]:.4f}   (expect 1.0)')
print(f'  ⟨R⟩  = {ss_c["mu_R"]:.2f}   (expect 50.0)')
print(f'  σ²_R = {ss_c["sigma_R"]**2:.2f}   (Poisson expect ~50)')
print(f'  Fano = {ss_c["sigma_R"]**2 / ss_c["mu_R"]:.3f}   (expect ~1.0)')
print(f'  Cov  = {ss_c["cov_RG"]:.5f}   (expect ~0.0)')

assert abs(ss_c['mu_G'] - 1.0) < 0.01
fano = ss_c['sigma_R']**2 / ss_c['mu_R']
assert abs(fano - 1.0) < 0.25, f'Fano = {fano:.3f}'
print('\n✅ Passed!')

In [ ]:
# Visual: constitutive expression
ana_c = analytical_steady_state(k_on=0.1, k_off=0.0, k_syn=10.0, k_deg=0.2)
_, fig_r, _ = show_sample_moments(m_const, analytical=ana_c)
fig_r.update_layout(title_text='<b>Edge case:</b> Constitutive (k<sub>off</sub>=0)')
fig_r.show()

### 3b. No degradation ($k_{\text{deg}} = 0$)

mRNA is never destroyed → $\langle R \rangle$ grows linearly:
$\langle R(t) \rangle \approx k_{\text{syn}} \cdot \mu_G \cdot t$

Expected: $\langle G \rangle \to 0.5$, slope $\approx 2.5$.

In [ ]:
data_nodeg = simulate_telegraph(
    k_on=0.5, k_off=0.5, k_syn=5.0, k_deg=0.0,
    t0=0.0, g0=1, r0=0,
    n_sim=3000, n_rep=500,
)
m_nodeg = compute_time_grid_moments(data_nodeg, n_points=500)

# Gene should equilibrate
tail = max(1, len(m_nodeg['time']) // 10)
mu_G_late = float(m_nodeg['mu_G'][-tail:].mean())

# R grows linearly: fit slope on late portion
t_late = m_nodeg['time'][-tail:]
R_late = m_nodeg['mu_R'][-tail:]
slope = np.polyfit(t_late, R_late, 1)[0]
expected_slope = 5.0 * 0.5  # k_syn * μ_G = 2.5

print('── No degradation (k_deg = 0) ──')
print(f'  ⟨G⟩_late = {mu_G_late:.4f}   (expect ~0.5)')
print(f'  dR/dt    = {slope:.3f}   (expect ~{expected_slope:.1f})')
print(f'  Final ⟨R⟩ = {m_nodeg["mu_R"][-1]:.0f}')

assert abs(mu_G_late - 0.5) < 0.08, f'⟨G⟩ = {mu_G_late}'
assert abs(slope - expected_slope) / expected_slope < 0.20, f'slope = {slope}'
print('\n✅ Passed!')

In [ ]:
# Visual: linear growth
fig = go.Figure(layout=_base_layout(
    title='<b>Edge case:</b> k<sub>deg</sub>=0 — mRNA grows linearly',
    ylabel='⟨R⟩ (molecules)',
    figsize=(5.5, 3.2),
))

fig.add_trace(go.Scatter(
    x=m_nodeg['time'], y=m_nodeg['mu_R'], mode='lines',
    line=dict(color=_RED, width=2.2),
    name='SSA ⟨R⟩',
))

t_ref = m_nodeg['time']
fig.add_trace(go.Scatter(
    x=t_ref, y=expected_slope * t_ref, mode='lines',
    line=dict(color='#94a3b8', width=1.5, dash='dash'),
    name=f'Expected: {expected_slope:.1f}·t',
))

fig.show()

---
## Summary

| Test | Method | Pass criterion | Status |
|---|---|---|---|
| **1. Convergence** | Time-grid SS avg vs analytical | Error ∝ $1/\sqrt{N}$ | ✅ |
| **2. Steady state** | 4 regimes, $N_{\text{rep}}=500$ | Rel. error < 15% | ✅ |
| **3a. $k_{\text{off}}=0$** | Constitutive: Fano → 1 | Poisson | ✅ |
| **3b. $k_{\text{deg}}=0$** | No degradation: linear growth | Slope = $k_{\text{syn}} \mu_G$ | ✅ |

> **Note:** The `compute_sample_moments` function averages at step
> indices, which over-samples high-propensity states ($G=1$). For
> unbiased validation, we used `compute_time_grid_moments` which
> interpolates trajectories to a common time grid.